# Translation

Parameters Definition

In [ ]:
source_lang, source_lang_iso = "spa", "spa"
target_lang, target_lang_iso = "guc", "guc" # or pbb, Paez
base_model = "t5-base" # or t5-small, t5-large, google/mt5-base, facebook/bart-large, etc

In [2]:
# Login to Hugging Face
from huggingface_hub import login
login()

In [3]:
# Login to Weights and Biases
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/adega/.netrc.
wandb: Currently logged in as: la-rodriguez (la-rodriguez-universidad-de-los-andes) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Dataset Processing

In [4]:
from datasets import load_dataset

dataset = load_dataset(f"lrodriguez22/translation_{source_lang_iso}_{target_lang_iso}")
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 76676
    })
    validation: Dataset({
        features: ['id', 'translation'],
        num_rows: 19170
    })
    test: Dataset({
        features: ['id', 'translation'],
        num_rows: 23962
    })
})

Then take a look at an example:

In [5]:
dataset["train"][100]

{'id': 34396,
 'translation': {'guc': 'piirakaa sümüin tü ayaakuaakat',
  'spa': 'vea la ilustracion del principio'}}

## Preprocess

In [6]:
from transformers import AutoTokenizer

checkpoint = base_model
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [7]:
prefix = f"translate {source_lang} to {target_lang}: "

def preprocess_function(examples):
    inputs = [prefix + example[source_lang] for example in examples["translation"]]
    targets = [example[target_lang] for example in examples["translation"]]
    model_inputs = tokenizer(inputs, text_target=targets, max_length=512, truncation=True)
    return model_inputs

In [8]:
print(dataset["train"][0])

{'id': 15577, 'translation': {'guc': "yüleeshaatasü ma'in", 'spa': 'esta muy fragil'}}


In [9]:
tokenized_dataset = dataset.map(preprocess_function, batched=True)

Now create a batch of examples using [DataCollatorForSeq2Seq](https://huggingface.co/docs/transformers/main/en/main_classes/data_collator#transformers.DataCollatorForSeq2Seq). It's more efficient to *dynamically pad* the sentences to the longest length in a batch during collation, instead of padding the whole dataset to the maximum length.

In [10]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=checkpoint)

## Evaluate

In [11]:
import numpy as np
import evaluate

metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]

    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

## Train

In [12]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

In [13]:
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

In [14]:
new_model_name = f'{base_model}-translation-{source_lang_iso}-{target_lang_iso}'

In [15]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
from transformers import EarlyStoppingCallback

training_args = Seq2SeqTrainingArguments(
    output_dir=f"./results/{new_model_name}",
    eval_strategy="epoch",
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    fp16=False,
    optim="adafactor",
    weight_decay=0.01,
    save_total_limit=5,
    num_train_epochs=10,
    predict_with_generate=True,
    push_to_hub=True,
    load_best_model_at_end = True,
    report_to="wandb",
    warmup_steps=10,
    logging_steps=1,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer.train()

In [17]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/lrodriguez22/t5-base-translation-spa-guc/commit/9507b938e3deaf50448ecc6099a72536c8071671', commit_message='End of training', commit_description='', oid='9507b938e3deaf50448ecc6099a72536c8071671', pr_url=None, repo_url=RepoUrl('https://huggingface.co/lrodriguez22/t5-base-translation-spa-guc', endpoint='https://huggingface.co', repo_type='model', repo_id='lrodriguez22/t5-base-translation-spa-guc'), pr_revision=None, pr_num=None)